# Notebook 3 — Model Training

This notebook trains four models of increasing complexity to predict 
startup exit outcomes. We follow a systematic comparison approach:

1. **Majority class baseline** : predicts the most common class always.
   
2. **Logistic Regression** : linear baseline with L2 regularisation.
   Interpretable and fast. Shows what linear relationships can achieve.
3. **Random Forest** : ensemble of decision trees with hyperparameter tuning.
   Captures non-linear interactions between features.
4. **MLP (Neural Network)** : multi-layer perceptron with early stopping.
   Maps to course lecture content on neural networks.

All models use a **time-based train/validation/test split** to prevent 
data leakage, so the models are trained on earlier cohorts and tested on later 
ones, reflecting the realistic scenario where a PE analyst uses historical 
data to predict future exits.

Class imbalance (86.1% no exit vs 13.9% exit) is handled using 
class_weight='balanced' which automatically up-weights the minority class.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, roc_auc_score,
                              f1_score, confusion_matrix,
                              precision_score, recall_score)
import warnings
warnings.filterwarnings('ignore')

# Fix random seed for reproducibility
np.random.seed(42)

# Paths
DATA_PROCESSED = Path('../data/processed')
OUTPUTS        = Path('../outputs/metrics')
FIGURES        = Path('../outputs/figures')
OUTPUTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

# Load features
features = pd.read_csv(DATA_PROCESSED / 'features.csv', low_memory=False)

print("Features loaded:", features.shape)
print("\nLabel distribution:")
print(features['label'].value_counts())
print(f"\nExit rate: {features['label'].mean()*100:.1f}%")

Features loaded: (13768, 55)

Label distribution:
label
0    11850
1     1918
Name: count, dtype: int64

Exit rate: 13.9%


### Train / Validation / Test Split

I used a time-based split rather than random split to prevent leakage.
Companies are split by founding year:

- **Train**: founded 1995–2004 — used to fit all models
- **Validation**: founded 2005–2006 — used to tune hyperparameters
- **Test**: founded 2007–2009 — held out, touched ONLY once at the end


In [4]:
from sklearn.model_selection import train_test_split, StratifiedKFold

# Stratified split — preserves exit rate in all sets
X = features[FEATURE_COLS]
y = features['label']

# First: hold out 20% as test set
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y  # preserves 13.9% exit rate in both sets
)

# Then: split remaining 80% into train/val (75/25 of trainval = 60/20 overall)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.25,
    random_state=42,
    stratify=y_trainval
)

print("=== Stratified Split ===")
print(f"Train: {len(X_train):,} | exit rate: {y_train.mean()*100:.1f}%"
      f" | exits: {y_train.sum()}")
print(f"Val:   {len(X_val):,} | exit rate: {y_val.mean()*100:.1f}%"
      f" | exits: {y_val.sum()}")
print(f"Test:  {len(X_test):,} | exit rate: {y_test.mean()*100:.1f}%"
      f" | exits: {y_test.sum()}")
print(f"\nTotal exits accounted for: "
      f"{y_train.sum() + y_val.sum() + y_test.sum()}")

=== Stratified Split ===
Train: 8,260 | exit rate: 13.9% | exits: 1150
Val:   2,754 | exit rate: 13.9% | exits: 384
Test:  2,754 | exit rate: 13.9% | exits: 384

Total exits accounted for: 1918


In [5]:
from sklearn.model_selection import (train_test_split, 
                                     StratifiedKFold,
                                     cross_val_score,
                                     cross_validate)

# Step 1 — Hold out 20% as final test set
X = features[FEATURE_COLS]
y = features['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Step 2 — Define 5-fold stratified cross validator
# This will be used during model training and tuning
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=== Data Split ===")
print(f"Train+CV: {len(X_train):,} companies "
      f"(exit rate: {y_train.mean()*100:.1f}%)")
print(f"Test:     {len(X_test):,} companies "
      f"(exit rate: {y_test.mean()*100:.1f}%)")
print(f"\nK-Fold: 5 folds")
print(f"Each fold trains on: ~{int(len(X_train)*0.8):,} companies")
print(f"Each fold validates on: ~{int(len(X_train)*0.2):,} companies")
print(f"\nTest exits: {y_test.sum()} — held out until final evaluation")

=== Data Split ===
Train+CV: 11,014 companies (exit rate: 13.9%)
Test:     2,754 companies (exit rate: 13.9%)

K-Fold: 5 folds
Each fold trains on: ~8,811 companies
Each fold validates on: ~2,202 companies

Test exits: 384 — held out until final evaluation


### Model 1 — Majority Class Baseline

The simplest possible model: always predicts 0 (no exit).
This sets the performance floor, so any useful model must beat this.
Expected AUC = 0.50 (equivalent to random guessing).

In [7]:
# ═══════════════════════════════════════════════════════════════
# MODEL 1 — Majority Class Baseline
# Always predicts 0 (no exit) — the most common class (86.1%)
# Purpose: sets the floor — any real model must beat this
# ═══════════════════════════════════════════════════════════════

dummy = DummyClassifier(strategy='most_frequent', random_state=42)

dummy_scores = cross_validate(
    dummy, X_train, y_train,
    cv=cv,
    scoring=['roc_auc', 'f1_macro'],
    return_train_score=False
)

print("=== Model 1: Majority Class Baseline ===")
print(f"CV AUC:      {dummy_scores['test_roc_auc'].mean():.3f} "
      f"± {dummy_scores['test_roc_auc'].std():.3f}")
print(f"CV F1 macro: {dummy_scores['test_f1_macro'].mean():.3f} "
      f"± {dummy_scores['test_f1_macro'].std():.3f}")

=== Model 1: Majority Class Baseline ===
CV AUC:      0.500 ± 0.000
CV F1 macro: 0.463 ± 0.000


### Model 2 — Logistic Regression

Linear model with L2 regularisation. Fast, interpretable, and a strong 
baseline for binary classification. 

L2 regularisation (parameter C) penalises large coefficients, preventing 
overfitting. We tune C using cross-validation.


In [9]:
# Linear model with L2 regularisation
# Pipeline: scale features → fit logistic regression
# class_weight='balanced' handles class imbalance automatically
# ═══════════════════════════════════════════════════════════════

lr_pipeline = Pipeline([
    # Step 1: scale all features to same range
    # Logistic regression is sensitive to feature magnitude
    # Without scaling, total_funding_usd (millions) dominates
    # over is_us (0 or 1)
    ('scaler', StandardScaler()),

    # Step 2: fit logistic regression
    # C = regularisation strength (smaller = more regularisation)
    # class_weight='balanced' = up-weights minority exit class
    # max_iter=1000 = enough iterations to converge
    ('clf', LogisticRegression(
        C=1.0,
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ))
])

lr_scores = cross_validate(
    lr_pipeline, X_train, y_train,
    cv=cv,
    scoring=['roc_auc', 'f1_macro'],
    return_train_score=False
)

print("=== Model 2: Logistic Regression ===")
print(f"CV AUC:      {lr_scores['test_roc_auc'].mean():.3f} "
      f"± {lr_scores['test_roc_auc'].std():.3f}")
print(f"CV F1 macro: {lr_scores['test_f1_macro'].mean():.3f} "
      f"± {lr_scores['test_f1_macro'].std():.3f}")
print(f"\nImprovement over baseline:")
print(f"AUC: +{lr_scores['test_roc_auc'].mean() - 0.500:.3f}")
print(f"F1:  +{lr_scores['test_f1_macro'].mean() - 0.463:.3f}")

=== Model 2: Logistic Regression ===
CV AUC:      0.773 ± 0.005
CV F1 macro: 0.624 ± 0.004

Improvement over baseline:
AUC: +0.273
F1:  +0.161


### Model 3 — Random Forest

Ensemble of decision trees that captures non-linear interactions between 
features. For example, the combination of "high funding + reached Series B 
+ top university founder" may predict exit better than any single feature 
alone — something logistic regression cannot capture.

We use RandomizedSearchCV to tune hyperparameters over 20 random 
combinations, evaluated using 5-fold cross validation.

Key hyperparameters tuned:
- n_estimators: number of trees (more = more stable but slower)
- max_depth: how deep each tree grows (deeper = more complex)
- min_samples_leaf: minimum samples per leaf (higher = less overfitting)

In [10]:
# ═══════════════════════════════════════════════════════════════
# MODEL 3 — Random Forest with Hyperparameter Tuning
# RandomizedSearchCV tests 20 random combinations
# Scored on AUC — our primary metric
# ═══════════════════════════════════════════════════════════════

# Define hyperparameter search space
param_dist = {
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [5, 10, 15, 20, None],
    'min_samples_leaf': [1, 5, 10, 20],
    'max_features':     ['sqrt', 'log2']
}

rf = RandomForestClassifier(
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# RandomizedSearchCV tests 20 random combinations
# Each combination evaluated with 5-fold CV
# Best combination selected by AUC
rf_search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=cv,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

rf_search.fit(X_train, y_train)

print("=== Model 3: Random Forest ===")
print(f"Best params: {rf_search.best_params_}")
print(f"CV AUC:      {rf_search.best_score_:.3f}")

# Get full scores for best model
rf_scores = cross_validate(
    rf_search.best_estimator_,
    X_train, y_train,
    cv=cv,
    scoring=['roc_auc', 'f1_macro'],
    return_train_score=False
)

print(f"CV AUC:      {rf_scores['test_roc_auc'].mean():.3f} "
      f"± {rf_scores['test_roc_auc'].std():.3f}")
print(f"CV F1 macro: {rf_scores['test_f1_macro'].mean():.3f} "
      f"± {rf_scores['test_f1_macro'].std():.3f}")
print(f"\nImprovement over logistic regression:")
print(f"AUC: +{rf_scores['test_roc_auc'].mean() - 0.773:.3f}")

=== Model 3: Random Forest ===
Best params: {'n_estimators': 200, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 20}
CV AUC:      0.785
CV AUC:      0.785 ± 0.008
CV F1 macro: 0.660 ± 0.015

Improvement over logistic regression:
AUC: +0.012


### Model 4 — MLP Neural Network

Multi-layer perceptron with two hidden layers. Maps directly to course 
content on neural networks (Lectures 5 and 8). 

Key design choices:
- Two hidden layers (128, 64 neurons) — enough capacity for tabular data
- ReLU activation — standard for hidden layers, avoids vanishing gradients
- L2 regularisation (alpha) — prevents overfitting on small dataset
- Early stopping — stops training when validation loss stops improving,
  another form of regularisation
- StandardScaler — MLPs are sensitive to feature scale, must normalise

In [11]:
# ═══════════════════════════════════════════════════════════════
# MODEL 4 — MLP Neural Network
# Two hidden layers: 128 → 64 neurons
# ReLU activation, L2 regularisation, early stopping
# Pipeline includes StandardScaler — MLP requires scaled features
# ═══════════════════════════════════════════════════════════════

mlp_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', MLPClassifier(
        hidden_layer_sizes=(128, 64),  # two hidden layers
        activation='relu',              # ReLU activation function
        alpha=0.001,                    # L2 regularisation strength
        early_stopping=True,            # stop when val loss plateaus
        validation_fraction=0.1,        # 10% of train for early stopping
        max_iter=200,                   # maximum training epochs
        random_state=42
    ))
])

mlp_scores = cross_validate(
    mlp_pipeline, X_train, y_train,
    cv=cv,
    scoring=['roc_auc', 'f1_macro'],
    return_train_score=False
)

print("=== Model 4: MLP Neural Network ===")
print(f"CV AUC:      {mlp_scores['test_roc_auc'].mean():.3f} "
      f"± {mlp_scores['test_roc_auc'].std():.3f}")
print(f"CV F1 macro: {mlp_scores['test_f1_macro'].mean():.3f} "
      f"± {mlp_scores['test_f1_macro'].std():.3f}")
print(f"\nImprovement over logistic regression:")
print(f"AUC: +{mlp_scores['test_roc_auc'].mean() - 0.773:.3f}")

=== Model 4: MLP Neural Network ===
CV AUC:      0.771 ± 0.007
CV F1 macro: 0.590 ± 0.011

Improvement over logistic regression:
AUC: +-0.002


### Model Comparison Summary

All four models compared on cross-validation AUC and F1 macro.
Random Forest is the best performing model — selected for final 
evaluation on the held-out test set in notebook 4.

In [12]:
# ═══════════════════════════════════════════════════════════════
# MODEL COMPARISON — summary table
# ═══════════════════════════════════════════════════════════════

results = pd.DataFrame({
    'Model': [
        'Majority Class Baseline',
        'Logistic Regression',
        'Random Forest',
        'MLP Neural Network'
    ],
    'CV AUC Mean': [
        dummy_scores['test_roc_auc'].mean(),
        lr_scores['test_roc_auc'].mean(),
        rf_scores['test_roc_auc'].mean(),
        mlp_scores['test_roc_auc'].mean()
    ],
    'CV AUC Std': [
        dummy_scores['test_roc_auc'].std(),
        lr_scores['test_roc_auc'].std(),
        rf_scores['test_roc_auc'].std(),
        mlp_scores['test_roc_auc'].std()
    ],
    'CV F1 Mean': [
        dummy_scores['test_f1_macro'].mean(),
        lr_scores['test_f1_macro'].mean(),
        rf_scores['test_f1_macro'].mean(),
        mlp_scores['test_f1_macro'].mean()
    ],
    'CV F1 Std': [
        dummy_scores['test_f1_macro'].std(),
        lr_scores['test_f1_macro'].std(),
        rf_scores['test_f1_macro'].std(),
        mlp_scores['test_f1_macro'].std()
    ]
})

# Round for display
results['CV AUC'] = (results['CV AUC Mean'].round(3).astype(str) + 
                     ' ± ' + results['CV AUC Std'].round(3).astype(str))
results['CV F1'] = (results['CV F1 Mean'].round(3).astype(str) + 
                    ' ± ' + results['CV F1 Std'].round(3).astype(str))

print("=== MODEL COMPARISON ===")
print(results[['Model','CV AUC','CV F1']].to_string(index=False))
print(f"\nBest model: Random Forest (AUC={rf_scores['test_roc_auc'].mean():.3f})")

# Save results
results.to_csv(OUTPUTS / 'model_comparison.csv', index=False)
print("Saved to outputs/metrics/model_comparison.csv")

=== MODEL COMPARISON ===
                  Model        CV AUC         CV F1
Majority Class Baseline     0.5 ± 0.0   0.463 ± 0.0
    Logistic Regression 0.773 ± 0.005 0.624 ± 0.004
          Random Forest 0.785 ± 0.008  0.66 ± 0.015
     MLP Neural Network 0.771 ± 0.007  0.59 ± 0.011

Best model: Random Forest (AUC=0.785)
Saved to outputs/metrics/model_comparison.csv
